In [17]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

In [4]:
CLASS_NAMES = [
    "T-shirt/top",  # 0
    "Trouser",      # 1
    "Pullover",     # 2
    "Dress",        # 3
    "Coat",         # 4
    "Sandal",       # 5
    "Shirt",        # 6
    "Sneaker",      # 7
    "Bag",          # 8
    "Ankle boot"    # 9
]

In [18]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize pixel values from [0, 255] to [0.0, 1.0]
X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten 28×28 images to 784-dimensional vectors (for Dense layers)
X_train_full = X_train_full.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

# Train / Validation / Test split
# 60,000 training images split into 48,000 train + 12,000 validation
VAL_SIZE = 12_000
X_val, y_val = X_train_full[:VAL_SIZE], y_train_full[:VAL_SIZE]
X_train, y_train = X_train_full[VAL_SIZE:], y_train_full[VAL_SIZE:]

print(f"Train samples : {X_train.shape[0]:,}")
print(f"Val samples : {X_val.shape[0]:,}")
print(f"Test samples : {X_test.shape[0]:,}")
print(f"Image shape : {X_train.shape[1]} features (28×28 flattened)")

Train samples : 48,000
Val samples : 12,000
Test samples : 10,000
Image shape : 784 features (28×28 flattened)


In [19]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Fashion-MNIST - One Sample Per Class", fontsize=15, fontweight="bold", y=1.02)

for i, ax in enumerate(axes.flat):
    # Grab the first training image that belongs to class i
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx].reshape(28, 28), cmap="gray_r")
    ax.set_title(CLASS_NAMES[i], fontsize=10, pad=6)
    ax.axis("off")

plt.tight_layout()
plt.savefig("sample_images.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: sample_images.png")

Saved: sample_images.png


In [7]:
model = keras.Sequential([
    layers.Input(shape=(784,)),

    # Hidden Layer 1
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(), # stabilises training
    layers.Dropout(0.3), # prevents overfitting

    # Hidden Layer 2
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    # Output Layer - softmax converts raw scores to probabilities
    layers.Dense(10, activation="softmax"),

], name="FashionNet")

model.summary()

Model: "FashionNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 236,682 (924.54 KB)

 Trainable params: 235,914 (921.54 KB)

 Non-trainable params: 768 (3.00 KB)

In [8]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.7961 - loss: 0.5730 - val_accuracy: 0.8346 - val_loss: 0.4575 - learning_rate: 0.0010
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8362 - loss: 0.4484 - val_accuracy: 0.8577 - val_loss: 0.3907 - learning_rate: 0.0010
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8483 - loss: 0.4192 - val_accuracy: 0.8608 - val_loss: 0.3850 - learning_rate: 0.0010
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8561 - loss: 0.3975 - val_accuracy: 0.8483 - val_loss: 0.4097 - learning_rate: 0.0010
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8627 - loss: 0.3792
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8601 - loss: 0.3817 - val_accuracy: 0.8558 - val_loss: 0.3885 - learning_rate: 0.0010
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8719 - loss: 0.3484 - 

In [20]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"  Test Accuracy : {test_acc * 100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")

  Test Accuracy : 88.71%
  Test Loss     : 0.3255


In [21]:
epochs_ran = range(1, len(history.history["loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History", fontsize=14, fontweight="bold")

# Loss
ax1.plot(epochs_ran, history.history["loss"], color="#3B82F6", lw=2, label="Train")
ax1.plot(epochs_ran, history.history["val_loss"], color="#EF4444", lw=2, ls="--", label="Validation")
ax1.set_title("Loss per Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(epochs_ran, history.history["accuracy"], color="#3B82F6", lw=2, label="Train")
ax2.plot(epochs_ran, history.history["val_accuracy"], color="#EF4444", lw=2, ls="--", label="Validation")
ax2.set_title("Accuracy per Epoch")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

Saved: training_curves.png


In [22]:
# Get model predictions on the test set
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

# Build the confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True) # row-normalised

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Proportion correct"}
)
ax.set_title(f"Confusion Matrix  (Test Accuracy = {test_acc*100:.1f}%)", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix.png")

Saved: confusion_matrix.png


In [23]:
per_class_acc = cm_norm.diagonal()

# Colour code: green ≥88%, yellow ≥80%, red <80%
colours = [
    "#10B981" if a >= 0.88 else "#F59E0B" if a >= 0.80 else "#EF4444"
    for a in per_class_acc
]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(CLASS_NAMES, per_class_acc * 100, color=colours, edgecolor="white", width=0.6)
ax.axhline(test_acc * 100, color="gray", ls="--", lw=1.5, label=f"Overall avg ({test_acc*100:.1f}%)")
ax.set_ylim(55, 105)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-Class Accuracy", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=35, labelsize=9)
ax.legend(fontsize=9)

for bar, val in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{val*100:.1f}%",
            ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("per_class_accuracy.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: per_class_accuracy.png")

Saved: per_class_accuracy.png


In [24]:
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0) # zero out correct predictions
top6 = np.argsort(cm_no_diag.flatten())[::-1][:6] # 6 worst confusion pairs

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
fig.suptitle("Top 6 Confusions - Images the Model Got Wrong", fontsize=12, fontweight="bold")

for ax, idx in zip(axes, top6):
    true_cls = idx // 10
    pred_cls = idx % 10
    mask = (y_test == true_cls) & (y_pred == pred_cls)
    if not mask.any():
        ax.axis("off")
        continue
    sample = X_test[mask][0].reshape(28, 28)
    ax.imshow(sample, cmap="gray_r")
    ax.set_title(f"True: {CLASS_NAMES[true_cls]}\nPred: {CLASS_NAMES[pred_cls]}", fontsize=7.5, color="#DC2626")
    ax.axis("off")

plt.tight_layout()
plt.savefig("top_confusions.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: top_confusions.png")

Saved: top_confusions.png


In [25]:
print(" Classification Report")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

 Classification Report
              precision    recall  f1-score   support

 T-shirt/top       0.82      0.87      0.84      1000
     Trouser       0.99      0.97      0.98      1000
    Pullover       0.78      0.83      0.80      1000
       Dress       0.90      0.90      0.90      1000
        Coat       0.80      0.82      0.81      1000
      Sandal       0.98      0.95      0.97      1000
       Shirt       0.75      0.65      0.69      1000
     Sneaker       0.93      0.96      0.95      1000
         Bag       0.98      0.97      0.97      1000
  Ankle boot       0.96      0.95      0.96      1000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [26]:
model.save("fashion_mnist_model.keras")
print("Model saved as fashion_mnist_model.keras")

Model saved as fashion_mnist_model.keras
